# Lab 2: Prompt Engineering for IT Infrastructure Operations Using OpenAI API

## Overview
In this lab, you will build a GenAI-powered assistant for IT infrastructure operations using the OpenAI API and apply prompt engineering techniques to improve response quality.

### Learning Objectives
- Understand Generative AI fundamentals and transformer architecture
- Apply prompt engineering techniques to IT operations
- Build a practical GenAI application for incident analysis and remediation
- Experiment with system messages, role definition, context injection, and tone settings
- Evaluate model performance on real-world IT scenarios

### Lab Structure
1. **Introduction to Generative AI**
2. **Transformer Architecture Deep Dive**
3. **OpenAI API Fundamentals**
4. **Prompt Engineering Strategies**
5. **Building the IT Infrastructure Assistant**
6. **Practical Exercises with Real Data**
7. **Quiz Assessment**

## Section 1: Introduction to Generative AI

### What is Generative AI?
Generative AI refers to artificial intelligence systems that can generate new content, such as text, images, or code, based on patterns learned from training data. Unlike discriminative AI (which classifies existing data), generative AI learns the underlying distribution of data and can create novel outputs.

### Key Characteristics
- **Autoregressive Processing**: Generates one token at a time, predicting the next token based on previous context
- **Probabilistic**: Outputs are based on probability distributions learned during training
- **Context-Aware**: Maintains understanding of conversation history and context
- **Few-Shot Learning**: Can adapt to new tasks with minimal examples

### Applications in IT Operations
- **Incident Analysis**: Analyzing logs and alerts to understand root causes
- **Log Interpretation**: Parsing complex log files to extract meaningful insights
- **Alert Triage**: Prioritizing and categorizing alerts by severity and impact
- **Root Cause Investigation**: Tracing issues to their source
- **Remediation Recommendation**: Suggesting fixes and preventive measures

## Section 2: Transformer Architecture

### The Transformer Model
The Transformer architecture, introduced in the paper "Attention is All You Need" (2017), revolutionized NLP by replacing recurrent networks with attention mechanisms.

### Core Components

#### 1. **Self-Attention Mechanism**
```
Attention(Q, K, V) = softmax(Q·K^T / √d_k)·V

Where:
- Q (Query): What are we looking for?
- K (Key): What information do we have?
- V (Value): What information should we return?
- d_k: Dimension of keys
```

#### 2. **Multi-Head Attention**
- Parallel attention heads capture different aspects of relationships
- Each head learns different representation subspaces
- Outputs concatenated and projected

#### 3. **Feed-Forward Networks**
- Two linear transformations with ReLU activation
- Applied position-wise to each token independently

#### 4. **Positional Encoding**
- Adds information about token positions in the sequence
- Uses sine and cosine functions with different frequencies

### Encoder-Decoder Architecture
- **Encoder**: Processes input sequence into rich representations
- **Decoder**: Generates output sequence token by token using encoder outputs
- **GPT Models**: Decoder-only architecture, predict next token in sequence

## Section 3: Setup and OpenAI API Fundamentals

### Installation and Configuration

First, install the OpenAI library and load your API key:

In [9]:
# Install required libraries
!pip install openai pandas numpy python-dotenv --quiet

import os
import json
import pandas as pd
import numpy as np
from openai import OpenAI
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# --- FIX: Set your OpenAI API key here ---
# Get your key from https://platform.openai.com/api-keys
# Uncomment the line below and replace 'your-key-here' with your actual key.
# Set your key in the .env file in the lab folder (see .env.example) - never paste it into a notebook
# ----------------------------------------

# Load API key from environment variable
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("⚠️  WARNING: OPENAI_API_KEY not set. Set it using: os.environ['OPENAI_API_KEY'] = 'your-key'")
else:
    print("✓ OpenAI API key loaded successfully")

client = OpenAI(api_key=api_key)

✓ OpenAI API key loaded successfully


In [10]:
# Load IT Infrastructure Data
df = pd.read_csv('it_infrastructure_data.csv')

print("IT Infrastructure Dataset Overview")
print("=" * 60)
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nBasic Statistics:")
print(df.describe())

IT Infrastructure Dataset Overview
Shape: (100, 11)

Columns: ['Timestamp', 'ServerID', 'AlertType', 'Severity', 'CPUUsage(%)', 'MemoryUsage(%)', 'DiskUsage(%)', 'NetworkLatency(ms)', 'ErrorMessage', 'Service', 'Status']

Data Types:
Timestamp             object
ServerID              object
AlertType             object
Severity              object
CPUUsage(%)            int64
MemoryUsage(%)         int64
DiskUsage(%)           int64
NetworkLatency(ms)     int64
ErrorMessage          object
Service               object
Status                object
dtype: object

First 5 rows:
             Timestamp      ServerID       AlertType Severity  CPUUsage(%)  \
0  2026-07-26 01:56:03  SRV-CACHE-01      HighMemory   Medium           83   
1  2026-07-26 02:06:03    SRV-WEB-01      HighMemory     Info           36   
2  2026-07-26 02:16:03     SRV-DB-01      HighMemory      Low           18   
3  2026-07-26 02:26:03    SRV-WEB-02  NetworkLatency     Info           18   
4  2026-07-26 02:36:03     S

## Section 4: Prompt Engineering Strategies for IT Operations

### 4.1 System Messages and Role Definition
The system message defines the AI's behavior, expertise, and constraints. It's crucial for establishing context.

In [11]:
# Define System Messages for Different Roles

SYSTEM_MESSAGES = {
    'incident_analyst': """You are an expert IT Infrastructure Incident Analyst with 15 years of experience.
Your role is to:
- Analyze infrastructure alerts and incidents systematically
- Identify root causes with high accuracy
- Provide clear, actionable remediation steps
- Consider system interdependencies and cascading failures
- Communicate findings in technical but accessible language

Always structure your response as:
1. INCIDENT SUMMARY
2. ROOT CAUSE ANALYSIS
3. SEVERITY ASSESSMENT
4. IMMEDIATE ACTIONS
5. LONG-TERM PREVENTION

Be precise, factual, and focus on measurable metrics.""",

    'log_interpreter': """You are an expert Log Analysis Specialist with deep knowledge of system logs, application logs, and infrastructure logs.
Your expertise:
- Parse and interpret complex log entries
- Identify error patterns and anomalies
- Extract actionable insights from verbose logs
- Connect related log entries across services
- Explain technical details in clear language

Always provide:
- Key findings (3-5 bullet points)
- Error classification (Critical/High/Medium/Low)
- Relevant metrics or timestamps
- Recommended next steps""",

    'alert_triage': """You are an expert Alert Triage Specialist responsible for prioritizing and routing infrastructure alerts.
Your responsibility:
- Classify alerts by actual business impact
- Identify false positives and suppress noise
- Route to appropriate teams
- Estimate time to resolution
- Consider service dependencies

Output format:
ALERT_ID | SEVERITY | IMPACT | AFFECTED_SERVICES | TEAM_ROUTE | ETA_RESOLUTION"""
}

print("System Message Templates Defined:")
for role, message in SYSTEM_MESSAGES.items():
    print(f"\n📋 {role.upper().replace('_', ' ')}")
    print("-" * 60)
    print(message[:200] + "...")

System Message Templates Defined:

📋 INCIDENT ANALYST
------------------------------------------------------------
You are an expert IT Infrastructure Incident Analyst with 15 years of experience.
Your role is to:
- Analyze infrastructure alerts and incidents systematically
- Identify root causes with high accurac...

📋 LOG INTERPRETER
------------------------------------------------------------
You are an expert Log Analysis Specialist with deep knowledge of system logs, application logs, and infrastructure logs.
Your expertise:
- Parse and interpret complex log entries
- Identify error patt...

📋 ALERT TRIAGE
------------------------------------------------------------
You are an expert Alert Triage Specialist responsible for prioritizing and routing infrastructure alerts.
Your responsibility:
- Classify alerts by actual business impact
- Identify false positives an...


### 4.2 Context Injection and Few-Shot Prompting
Injecting relevant context and examples significantly improves model accuracy.

In [12]:
# Context Templates for Different Scenarios

CONTEXT_TEMPLATES = {
    'infrastructure_baseline': """Current Infrastructure Baseline:
- Normal CPU usage: 40-60%
- Normal Memory usage: 50-70%
- Normal Disk usage: 60-75%
- Normal Network Latency: 10-30ms
- Normal Error Rate: <0.1%
- Expected Response Time: <500ms
""",

    'service_dependencies': """Service Dependency Map:
- API Service depends on: Database, Cache, Authentication
- WebServer depends on: LoadBalancer, API Service
- Database depends on: Storage, Network
- Cache depends on: Memory, Network
- PaymentGateway depends on: API, Security, Network
""",

    'escalation_policy': """Escalation Policy:
- Critical (Customer impact, Revenue at risk): Escalate immediately to Senior DBA/Architect
- High (Service degradation, Performance impact): Escalate to Team Lead within 15 min
- Medium (Minor issues, Limited impact): Team resolution, update every 30 min
- Low (Informational): Log and monitor, resolve within 24 hours
"""
}

# Few-shot Examples for Incident Analysis
INCIDENT_EXAMPLES = [
    {
        'incident': 'CPU spike to 95%, API response times degraded to 5s',
        'root_cause': 'Unoptimized query in payment processing triggered thundering herd of connections',
        'action': 'Kill long-running query, optimize SQL, implement connection pooling'
    },
    {
        'incident': 'Database replication lag exceeded 30s, write operations delayed',
        'root_cause': 'Network saturation due to large backup operation running during peak hours',
        'action': 'Stop backup, reschedule to off-peak, implement bandwidth management'
    },
    {
        'incident': 'Cache hit rate dropped from 98% to 45%, CPU usage increased',
        'root_cause': 'Cache restart lost all data, application fallback to database queries',
        'action': 'Warm cache manually, investigate restart cause, implement cache persistence'
    }
]

print("✓ Context templates and few-shot examples defined")
print(f"\nExample incident for reference:")
print(f"Incident: {INCIDENT_EXAMPLES[0]['incident']}")
print(f"Root Cause: {INCIDENT_EXAMPLES[0]['root_cause']}")
print(f"Action: {INCIDENT_EXAMPLES[0]['action']}")

✓ Context templates and few-shot examples defined

Example incident for reference:
Incident: CPU spike to 95%, API response times degraded to 5s
Root Cause: Unoptimized query in payment processing triggered thundering herd of connections
Action: Kill long-running query, optimize SQL, implement connection pooling


## Section 5: Building the IT Infrastructure Assistant

### 5.1 Creating API Helper Functions

In [13]:
class ITOperationsAssistant:
    """GenAI-powered IT Infrastructure Operations Assistant"""

    def __init__(self, api_key, model="gpt-3.5-turbo"):
        self.client = OpenAI(api_key=api_key)
        self.model = model
        self.conversation_history = []
        self.cost_tracking = {'input_tokens': 0, 'output_tokens': 0}

    def call_api(self, system_message, user_message, temperature=0.7, max_tokens=1000):
        """Call OpenAI API with prompt engineering best practices"""
        try:
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": user_message}
                ],
                temperature=temperature,
                max_tokens=max_tokens
            )

            # Track tokens for cost analysis
            self.cost_tracking['input_tokens'] += response.usage.prompt_tokens
            self.cost_tracking['output_tokens'] += response.usage.completion_tokens

            return response.choices[0].message.content
        except Exception as e:
            return f"Error: {str(e)}"

    def analyze_incident(self, incident_data):
        """Analyze an infrastructure incident"""
        system_msg = SYSTEM_MESSAGES['incident_analyst']

        # Build context with infrastructure baseline
        context = CONTEXT_TEMPLATES['infrastructure_baseline']
        context += CONTEXT_TEMPLATES['service_dependencies']
        context += CONTEXT_TEMPLATES['escalation_policy']

        # Add few-shot examples
        examples_text = "\nPast incident examples:\n"
        for i, example in enumerate(INCIDENT_EXAMPLES, 1):
            examples_text += f"{i}. Incident: {example['incident']}\n"
            examples_text += f"   Root Cause: {example['root_cause']}\n"
            examples_text += f"   Action: {example['action']}\n\n"

        user_msg = f"{context}\n{examples_text}\nAnalyze this current incident:\n{incident_data}"

        return self.call_api(system_msg, user_msg, temperature=0.5)

    def interpret_logs(self, log_data):
        """Interpret and analyze system logs"""
        system_msg = SYSTEM_MESSAGES['log_interpreter']

        context = "Analyze these logs for errors, patterns, and anomalies. "
        context += "Provide key findings, severity level, and recommended actions.\n\n"

        user_msg = context + log_data

        return self.call_api(system_msg, user_msg, temperature=0.3)

    def triage_alerts(self, alerts_data):
        """Triage and prioritize infrastructure alerts"""
        system_msg = SYSTEM_MESSAGES['alert_triage']

        context = CONTEXT_TEMPLATES['infrastructure_baseline']
        context += CONTEXT_TEMPLATES['service_dependencies']

        user_msg = f"{context}\nTriage these alerts (output as CSV format):\n{alerts_data}"

        return self.call_api(system_msg, user_msg, temperature=0.2)

    def get_cost_report(self):
        """Generate API usage and cost report"""
        input_tokens = self.cost_tracking['input_tokens']
        output_tokens = self.cost_tracking['output_tokens']

        # Approximate pricing for gpt-3.5-turbo (check current pricing)
        input_cost = (input_tokens / 1000) * 0.0005
        output_cost = (output_tokens / 1000) * 0.0015
        total_cost = input_cost + output_cost

        return {
            'input_tokens': input_tokens,
            'output_tokens': output_tokens,
            'total_tokens': input_tokens + output_tokens,
            'estimated_cost': f"${total_cost:.4f}"
        }

print("✓ ITOperationsAssistant class defined successfully")
print("\nClass Methods:")
print("- analyze_incident(): Analyze infrastructure incidents")
print("- interpret_logs(): Parse and interpret logs")
print("- triage_alerts(): Prioritize and route alerts")
print("- get_cost_report(): Track API usage and costs")

✓ ITOperationsAssistant class defined successfully

Class Methods:
- analyze_incident(): Analyze infrastructure incidents
- interpret_logs(): Parse and interpret logs
- triage_alerts(): Prioritize and route alerts
- get_cost_report(): Track API usage and costs


### 5.2 Initialize the Assistant

In [14]:
# Initialize the assistant
if api_key:
    assistant = ITOperationsAssistant(api_key=api_key, model="gpt-3.5-turbo")
    print("✓ IT Operations Assistant initialized")
else:
    assistant = None
    print("⚠️  Assistant not initialized - API key required")

✓ IT Operations Assistant initialized


## Section 6: Practical Exercises with Real Data

### Exercise 1: Incident Analysis
Analyze a critical infrastructure incident using prompt engineering best practices.

In [15]:
# Exercise 1: Analyze a critical incident

# Select a critical incident from the dataset
critical_incidents = df[df['Severity'] == 'Critical']

if len(critical_incidents) > 0:
    incident = critical_incidents.iloc[0]

    incident_summary = f"""Alert ID: {incident.name}
Timestamp: {incident['Timestamp']}
Server: {incident['ServerID']}
Service: {incident['Service']}
Alert Type: {incident['AlertType']}
Severity: {incident['Severity']}
Status: {incident['Status']}

Current Metrics:
- CPU Usage: {incident['CPUUsage(%)']}%
- Memory Usage: {incident['MemoryUsage(%)']}%
- Disk Usage: {incident['DiskUsage(%)']}%
- Network Latency: {incident['NetworkLatency(ms)']}ms

Error Message: {incident['ErrorMessage']}
"""

    print("INCIDENT DETAILS:")
    print("=" * 60)
    print(incident_summary)

    # Analyze using the assistant (if available)
    if assistant:
        print("\nANALYSIS FROM ASSISTANT:")
        print("=" * 60)
        analysis = assistant.analyze_incident(incident_summary)
        print(analysis)
    else:
        print("\n⚠️  Assistant not available - provide API key to see analysis")
else:
    print("No critical incidents in dataset")

INCIDENT DETAILS:
Alert ID: 11
Timestamp: 2026-07-26 03:46:03
Server: SRV-DB-01
Service: WebServer
Alert Type: ServiceDown
Severity: Critical
Status: Resolved

Current Metrics:
- CPU Usage: 92%
- Memory Usage: 66%
- Disk Usage: 65%
- Network Latency: 126ms

Error Message: Disk space below 5%


ANALYSIS FROM ASSISTANT:
1. INCIDENT SUMMARY:
On 2026-07-26 at 03:46:03, a Critical alert was triggered on server SRV-DB-01 for the WebServer service being down due to low disk space.

2. ROOT CAUSE ANALYSIS:
Root Cause: The WebServer service went down due to disk space falling below 5%, causing the server to be unable to function properly.

3. SEVERITY ASSESSMENT:
Severity: Critical, as the WebServer being down impacts customer access and can lead to revenue loss.

4. IMMEDIATE ACTIONS:
Immediate Actions:
- Free up disk space by deleting unnecessary files or logs.
- Investigate and clean up any temporary files or caches taking up space.
- Monitor disk space closely to ensure it does not reach cr

### Exercise 2: Alert Triage
Triage multiple alerts and prioritize by impact.

In [16]:
# Exercise 2: Triage multiple alerts

# Select various severity alerts
sample_alerts = df.groupby('Severity').head(2)

alerts_text = "Alert Summary:\n"
for idx, alert in sample_alerts.iterrows():
    alerts_text += f"ID:{idx} | Server:{alert['ServerID']} | Type:{alert['AlertType']} | Severity:{alert['Severity']} | Service:{alert['Service']} | Error:{alert['ErrorMessage'][:40]}...\n"

print("ALERTS TO TRIAGE:")
print("=" * 60)
print(alerts_text)

if assistant:
    print("\nTRIAGE RESULTS:")
    print("=" * 60)
    triage_result = assistant.triage_alerts(alerts_text)
    print(triage_result)
else:
    print("\n⚠️  Assistant not available")

ALERTS TO TRIAGE:
Alert Summary:
ID:0 | Server:SRV-CACHE-01 | Type:HighMemory | Severity:Medium | Service:Database | Error:High error rate detected...
ID:1 | Server:SRV-WEB-01 | Type:HighMemory | Severity:Info | Service:WebServer | Error:Database replication lag > 10s...
ID:2 | Server:SRV-DB-01 | Type:HighMemory | Severity:Low | Service:PaymentGateway | Error:Database replication lag > 10s...
ID:3 | Server:SRV-WEB-02 | Type:NetworkLatency | Severity:Info | Service:PaymentGateway | Error:High error rate detected...
ID:4 | Server:SRV-DB-02 | Type:HighMemory | Severity:High | Service:Monitoring | Error:Disk space below 5%...
ID:7 | Server:SRV-APP-02 | Type:HighMemory | Severity:High | Service:Authentication | Error:SSL certificate expiration warning...
ID:8 | Server:SRV-APP-02 | Type:ConnectionPool | Severity:Low | Service:API | Error:Connection timeout to database...
ID:9 | Server:SRV-CACHE-01 | Type:HighCPU | Severity:Medium | Service:PaymentGateway | Error:Connection timeout to databas

### Exercise 3: Prompt Engineering Experimentation
Compare different prompt strategies and their effectiveness.

In [17]:
# Exercise 3: Experiment with different prompt engineering techniques

def compare_prompts(incident_data):
    """Compare different prompt strategies"""

    strategies = {
        'basic': {
            'system': 'You are a helpful IT support assistant.',
            'temperature': 0.7
        },
        'role_defined': {
            'system': 'You are an expert IT infrastructure engineer with 20 years of experience analyzing production incidents.',
            'temperature': 0.5
        },
        'detailed_role': {
            'system': SYSTEM_MESSAGES['incident_analyst'],
            'temperature': 0.4
        }
    }

    results = {}

    if not assistant:
        print("⚠️  Assistant not available - API key required")
        return results

    for strategy_name, strategy_config in strategies.items():
        print(f"\n📊 Strategy: {strategy_name.upper()}")
        print("-" * 60)

        response = assistant.call_api(
            system_message=strategy_config['system'],
            user_message=f"Analyze this incident:\n{incident_data}",
            temperature=strategy_config['temperature'],
            max_tokens=500
        )

        print(response[:300] + "...")
        results[strategy_name] = response

    return results

if len(critical_incidents) > 0:
    incident = critical_incidents.iloc[0]
    incident_summary = f"Server {incident['ServerID']} experienced {incident['AlertType']} alert at {incident['Timestamp']}. Error: {incident['ErrorMessage']}"

    print("PROMPT ENGINEERING COMPARISON")
    print("=" * 60)
    print(f"Incident: {incident_summary}\n")

    results = compare_prompts(incident_summary)
else:
    print("No incidents available for comparison")

PROMPT ENGINEERING COMPARISON
Incident: Server SRV-DB-01 experienced ServiceDown alert at 2026-07-26 03:46:03. Error: Disk space below 5%


📊 Strategy: BASIC
------------------------------------------------------------
Based on the incident report, the server SRV-DB-01 experienced a ServiceDown alert due to the disk space falling below 5%. This means that the server's disk space has reached a critically low level, which can lead to performance issues, service disruptions, or even system crashes.

To address this i...

📊 Strategy: ROLE_DEFINED
------------------------------------------------------------
Based on the incident details provided, the server SRV-DB-01 experienced a ServiceDown alert triggered by low disk space, specifically below 5%. This type of incident can have several implications and potential causes:

1. **Root Cause Analysis**: The immediate cause of the ServiceDown alert is the ...

📊 Strategy: DETAILED_ROLE
-----------------------------------------------------------

### Exercise 4: Data Analysis and Metrics

In [18]:
# Exercise 4: Analyze patterns in the infrastructure data

print("INFRASTRUCTURE DATA ANALYSIS")
print("=" * 60)

# Alert type distribution
print("\n1. Alert Type Distribution:")
alert_dist = df['AlertType'].value_counts()
print(alert_dist)

# Severity distribution
print("\n2. Severity Level Distribution:")
severity_dist = df['Severity'].value_counts()
print(severity_dist)

# Service-wise incidents
print("\n3. Incidents by Service:")
service_incidents = df['Service'].value_counts()
print(service_incidents)

# High resource usage incidents
print("\n4. High Resource Usage (>80%):")
high_cpu = len(df[df['CPUUsage(%)'] > 80])
high_memory = len(df[df['MemoryUsage(%)'] > 80])
high_disk = len(df[df['DiskUsage(%)'] > 80])
print(f"High CPU: {high_cpu} incidents")
print(f"High Memory: {high_memory} incidents")
print(f"High Disk: {high_disk} incidents")

# Status distribution
print("\n5. Incident Resolution Status:")
status_dist = df['Status'].value_counts()
print(status_dist)

# Average metrics by severity
print("\n6. Average Metrics by Severity:")
severity_metrics = df.groupby('Severity')[['CPUUsage(%)', 'MemoryUsage(%)', 'DiskUsage(%)', 'NetworkLatency(ms)']].mean()
print(severity_metrics.round(2))

INFRASTRUCTURE DATA ANALYSIS

1. Alert Type Distribution:
AlertType
DiskSpace         16
HighMemory        15
ErrorRate         15
NetworkLatency    13
ServiceDown       13
HighCPU           12
ConnectionPool     8
ResponseTime       8
Name: count, dtype: int64

2. Severity Level Distribution:
Severity
Medium      23
Info        22
High        19
Low         18
Critical    18
Name: count, dtype: int64

3. Incidents by Service:
Service
API               16
WebServer         15
Authentication    15
PaymentGateway    13
Cache             12
LoadBalancer      11
Database          10
Monitoring         8
Name: count, dtype: int64

4. High Resource Usage (>80%):
High CPU: 18 incidents
High Memory: 13 incidents
High Disk: 26 incidents

5. Incident Resolution Status:
Status
Escalated     27
Resolved      25
InProgress    25
Unresolved    23
Name: count, dtype: int64

6. Average Metrics by Severity:
          CPUUsage(%)  MemoryUsage(%)  DiskUsage(%)  NetworkLatency(ms)
Severity                

### Exercise 5: API Cost Analysis

In [19]:
# Exercise 5: Track and analyze API costs

if assistant:
    cost_report = assistant.get_cost_report()

    print("API USAGE AND COST REPORT")
    print("=" * 60)
    print(f"Input Tokens: {cost_report['input_tokens']:,}")
    print(f"Output Tokens: {cost_report['output_tokens']:,}")
    print(f"Total Tokens: {cost_report['total_tokens']:,}")
    print(f"Estimated Cost: {cost_report['estimated_cost']}")
    print("\nNote: Actual costs may vary based on current pricing.")
    print("Check OpenAI pricing page for latest rates.")
else:
    print("No API calls made - initialize assistant with valid API key")

API USAGE AND COST REPORT
Input Tokens: 1,461
Output Tokens: 1,352
Total Tokens: 2,813
Estimated Cost: $0.0028

Note: Actual costs may vary based on current pricing.
Check OpenAI pricing page for latest rates.


## Section 7: Quiz Assessment

Test your understanding of GenAI, transformer architecture, and prompt engineering.

In [20]:
# Quiz Assessment

QUIZ_QUESTIONS = [
    {
        'id': 1,
        'question': 'What is the primary mechanism that allows transformers to process sequences in parallel?',
        'options': [
            'A) Recurrent connections',
            'B) Self-attention mechanism',
            'C) Convolutional filters',
            'D) LSTM gates'
        ],
        'correct': 'B'
    },
    {
        'id': 2,
        'question': 'In the attention formula Attention(Q,K,V), what does the temperature parameter control?',
        'options': [
            'A) The number of attention heads',
            'B) The randomness/creativity of responses',
            'C) The maximum sequence length',
            'D) The batch size for training'
        ],
        'correct': 'B'
    },
    {
        'id': 3,
        'question': 'What is the primary benefit of few-shot prompting in IT operations?',
        'options': [
            'A) Reduces API costs',
            'B) Provides examples for the model to learn from',
            'C) Eliminates the need for system messages',
            'D) Increases token limit'
        ],
        'correct': 'B'
    },
    {
        'id': 4,
        'question': 'Which prompt engineering technique is most effective for incident root cause analysis?',
        'options': [
            'A) Random sampling',
            'B) Chain-of-thought with context injection',
            'C) Single-word prompts',
            'D) No system message'
        ],
        'correct': 'B'
    },
    {
        'id': 5,
        'question': 'What is the role of positional encoding in transformer models?',
        'options': [
            'A) Compress the model size',
            'B) Provide information about token positions in sequences',
            'C) Speed up GPU computation',
            'D) Reduce overfitting'
        ],
        'correct': 'B'
    }
]

print("\n" + "=" * 60)
print("QUIZ: Generative AI and Prompt Engineering")
print("=" * 60)
print(f"\nTotal Questions: {len(QUIZ_QUESTIONS)}")
print("\nInstructions: Select the correct answer for each question.\n")

for q in QUIZ_QUESTIONS:
    print(f"Question {q['id']}: {q['question']}")
    for option in q['options']:
        print(f"  {option}")
    print(f"\n  → Correct Answer: {q['correct']}")
    print()


QUIZ: Generative AI and Prompt Engineering

Total Questions: 5

Instructions: Select the correct answer for each question.

Question 1: What is the primary mechanism that allows transformers to process sequences in parallel?
  A) Recurrent connections
  B) Self-attention mechanism
  C) Convolutional filters
  D) LSTM gates

  → Correct Answer: B

Question 2: In the attention formula Attention(Q,K,V), what does the temperature parameter control?
  A) The number of attention heads
  B) The randomness/creativity of responses
  C) The maximum sequence length
  D) The batch size for training

  → Correct Answer: B

Question 3: What is the primary benefit of few-shot prompting in IT operations?
  A) Reduces API costs
  B) Provides examples for the model to learn from
  C) Eliminates the need for system messages
  D) Increases token limit

  → Correct Answer: B

Question 4: Which prompt engineering technique is most effective for incident root cause analysis?
  A) Random sampling
  B) Chain-

## Summary and Key Takeaways

### What You've Learned

1. **Generative AI Fundamentals**
   - Autoregressive generation and probabilistic modeling
   - Applications in IT operations and incident management

2. **Transformer Architecture**
   - Self-attention mechanisms for parallel processing
   - Multi-head attention for diverse representations
   - Positional encoding for sequence understanding

3. **Prompt Engineering Best Practices**
   - Defining clear roles and system messages
   - Injecting relevant context and baselines
   - Using few-shot examples for accurate predictions
   - Controlling temperature for consistency vs. creativity

4. **Practical Implementation**
   - Building GenAI-powered IT operations assistant
   - Analyzing incidents with structured outputs
   - Triaging and prioritizing alerts
   - Tracking API usage and costs

### Next Steps
- Experiment with different temperature values (0.0 = deterministic, 1.0 = creative)
- Try different model variants (GPT-4, GPT-3.5-turbo, etc.)
- Implement additional prompting techniques (Chain-of-Thought, ReAct)
- Build feedback loops to evaluate and improve assistant performance
- Integrate with actual monitoring and alerting systems

### Resources
- OpenAI API Documentation: https://platform.openai.com/docs
- Prompt Engineering Guide: https://www.promptingguide.ai
- Transformer Papers: "Attention is All You Need" (Vaswani et al., 2017)
- Model Context Protocol (MCP) for tool integration